In [20]:
import pandas as pd, matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

# 1. Clean & Merge
s, p = pd.read_csv('output/sales_transactions_cleaned.csv'), pd.read_csv('products.csv')

# ลบช่องว่างในชื่อ Category (ป้องกันปัญหาหา data ไม่เจอ)
p['category'] = p['category'].str.strip().replace({'Pastry': 'Pastries'})

for c in ['price', 'cost']:
    p[c] = pd.to_numeric(p[c].astype(str).str.replace(r'[^0-9.]', '', regex=True), errors='coerce').abs()

tq = s.assign(rev=s.quantity * s.price - s.discount_amount.fillna(0)).merge(p, on='product_id')

# 2. Agg
# ใช้ .str.strip() อีกรอบตอนกรองเพื่อให้ชัวร์ว่าสะกดตรงเป้า
cats = ['Pastries', 'Bread', 'Tarte']
cr = tq[tq.category.str.strip().isin(cats)].groupby('category').rev.sum().reindex(cats).fillna(0)

t3 = tq.groupby('product_name')[['quantity','rev']].sum().nlargest(3, 'quantity').reset_index()
t3['rev'] = t3.rev.map('${:,.2f}'.format)

# 3. PDF
with PdfPages('output_z/Session1_ProductPerformance.pdf') as pdf:
    # หน้า 1: Bar Chart (ปรับ figsize ให้กว้างขึ้น และใช้ tight_layout)
    fig, ax = plt.subplots(figsize=(10, 6))
    cr.plot(kind='bar', color=['#E05C2A', '#2A7AE0', '#27A86A'], ax=ax, rot=0)
    ax.set(title='Revenue by Category', ylabel='Revenue ($)')
    ax.grid(axis='y', ls='--', alpha=0.5)
    
    fig.tight_layout() # จัดระเบียบไม่ให้ตัวหนังสือหลุดขอบ
    pdf.savefig(fig, bbox_inches='tight'); plt.close()

    # หน้า 2: Table (ขยาย figsize และลด scale เพื่อไม่ให้ตารางแหว่ง)
    fig, ax = plt.subplots(figsize=(10, 2)); ax.axis('off')
    tbl = ax.table(cellText=t3.values, colLabels=['Name', 'Qty', 'Rev'], loc='center', cellLoc='center')
    
    tbl.auto_set_font_size(False); tbl.set_fontsize(11)
    tbl.scale(1.2, 2.5) # ปรับ scale ให้พอดีกับหน้ากระดาษ
    
    for c in range(3): 
        tbl[0,c].set_facecolor('#E05C2A') 
        tbl[0,c].set_text_props(color='white', weight='bold')
        
    pdf.savefig(fig, bbox_inches='tight'); plt.close()

print('✅Saved!')

✅Saved!
